In [ ]:
import numpy as np
import pandas as pd


In [ ]:
df = pd.read_csv(r"NetFlix.csv\NetFlix.csv")

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().sum() / len(df)) * 100

In [ ]:
df[df['director'].isnull()]

In [ ]:
df[['director', 'cast','country']] = df[['director','cast','country']].fillna('Unknown')

df = df.dropna(subset=['date_added','rating'])

In [ ]:
df.isnull().sum()

In [ ]:
df.to_csv('netflix_titles_cleaned.csv', index=False)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
from sklearn.preprocessing import LabelEncoder

X = pd.concat([df['release_year'], df['duration']], axis=1)
y = df['type']

le = LabelEncoder()
y = le.fit_transform(y)

X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

lr = LogisticRegression()
lr.fit(X_train,y_train)
y_pred = lr.predict(X_test)
accuracy = accuracy_score(y_test,y_pred)
print(f"accuracy : {accuracy}")
print(f"Classification report:\n{classification_report(y_test,y_pred,target_names=le.classes_)}")


In [ ]:
cate = pd.get_dummies(df[['rating', 'country']],drop_first=True)
X = pd.concat([df['release_year'],cate],axis=1)
y = df['type']

y = le.fit_transform(y)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

lr = LogisticRegression(class_weight='balanced',max_iter=1000)
lr.fit(X_train,y_train)
y_pred = lr.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)
print(f"accuracy : {accuracy}")
print(classification_report(y_test,y_pred,target_names=le.classes_))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100,class_weight='balanced',random_state=42)
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)
print(f"accuracy : {accuracy}")
print(classification_report(y_test,y_pred,target_names=le.classes_))

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)

top_10 = importances.nlargest(10)

print(top_10)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)

y_pred = dt.predict(X_test)

for depth in [1,2,3,5,7,10]:
    dt_purned = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt_purned.fit(X_train,y_train)

    train_acc = dt_purned.score(X_train,y_train)
    test_acc = dt_purned.score(X_test, y_test)

    print(f"Depth {depth:2d} , train: {train_acc:.4f} , test: {test_acc:.4f}")

print(f"{accuracy_score(y_test,y_pred)}")
print(f"Train Accuracy: {dt.score(X_train,y_train):.4f}")
print(f"Test Accuracy: {dt.score(X_test,y_test):.4f}")

In [ ]:
scores = cross_val_score(DecisionTreeClassifier(max_depth=7,random_state=42), X,y,cv=5,scoring='accuracy')
print(f'{scores}')
print(f"mean accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3,5,7,10,15],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid,cv=5,scoring='accuracy')

grid.fit(X,y)
print(grid.best_params_)
print(f"Best cv accuracy: {grid.best_score_:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [50,100,200],  # Number of trees in the forest
    'max_depth': [5,10,15,20,None], # Depth of each tree
    'min_samples_split': [2,5,10],  # Minimum samples required to split an internal node
    'min_samples_leaf': [1,2,4],    # Minimum samples required at a leaf node
    'class_weight': [None,'balanced'] # Handling class imbalance
}

rf_random = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1    # Uses all available CPU cores for speed
)

rf_random.fit(X,y)

print("Best Parameters: ", rf_random.best_params_)
print(f"Best CV Accuracy: {rf_random.best_score_:.4f}")

In [ ]:
best_rf = rf_random.best_estimator_

y_pred = best_rf.predict(X_test)

print(f"accuracy {accuracy_score(y_test,y_pred)}")
print(f"classification_report: \n{classification_report(y_test,y_pred,target_names=le.classes_)}")


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ('scaler', StandardScaler()), # Normalizes continuous features like release_year
    ('rf', RandomForestClassifier(n_estimators=200, min_samples_split=10,random_state=42,n_jobs=-1))
])

# Fit ONLY on X_train to prevent test data leakage
pipeline.fit(X_train,y_train)

pipe_acc = pipeline.score(X_test,y_test)
print(f"Pipeline Test Accuracy: {pipe_acc:.4f}")